# Agentic AI Bootcamp
## Day 3 — Multi-Agent Systems, LangGraph & A2A Communication

---

**Course:** 5-Day Agentic AI Bootcamp  
**Day:** 3 of 5  
**Duration:** 2 hours  
**Platform:** University Assignment Platform — Multi-Agent Edition

---

### What you will build today
By the end of this session you will have a **5-agent pipeline** where:
- An **Orchestrator** classifies your question and routes it
- A **Planner** breaks it into research subtasks
- A **Researcher** (powered by the Day 2 ReAct loop) gathers information
- A **Writer** synthesises a clear answer
- A **Critic** scores and can loop the Writer until quality is high enough
- Agents **talk to each other** via a shared state dict (A2A communication)

### The 5-day arc
| Day | What we add | Status |
|-----|-------------|--------|
| Day 1 | Single reasoning agent with CoT | ✅ Done |
| Day 2 | Tool use — ReAct loop (web search, calculator, file reader) | ✅ Done |
| **Day 3** | **Multi-agent system with LangGraph + A2A communication** | ← you are here |
| Day 4 | Persistent memory (ChromaDB/RAG) + MCP tools | Coming next |
| Day 5 | FastAPI + WhatsApp interface + Docker + monitoring | Coming next |

> **Instructor tip:** Day 3 is the conceptual leap. Before touching code, spend 5 minutes
> mentally mapping each agent to a person on a team. The Orchestrator is the manager.
> The Planner writes the project plan. The Researcher goes to the library.
> The Writer drafts the report. The Critic is the editor. Same roles, different code.

## Today's Agenda

| Time | Topic |
|------|-------|
| 0:00 – 0:10 | Setup + quick review of Day 2 ReAct |
| 0:10 – 0:25 | **Part 1:** ReAct recap — tool use in one cell |
| 0:25 – 0:35 | **Part 1b:** Using hosted tools — you don't need to build everything |
| 0:35 – 0:50 | **Part 2:** Why multi-agent? The specialisation argument |
| 0:50 – 1:05 | **Part 3:** LangGraph basics — state, nodes, edges |
| 1:05 – 1:15 | **Part 4:** Conditional routing and critic loops |
| 1:15 – 1:25 | **Part 5:** Agent handoffs and context preservation |
| 1:25 – 1:45 | **Part 6:** Lab — full 5-agent pipeline |
| 1:45 – 1:55 | **Part 7:** A2A — tool delegation, RPC, and communication patterns |
| 1:55 – 2:00 | **Part 8:** Day 4 preview — memory, RAG, and what's next |

---
# Setup
Run the two cells below first. We add `langgraph` to the Day 1+2 dependencies.

In [8]:
# ── Install all required packages ─────────────────────────────────────────────
# Run this cell once before anything else.

!pip install anthropic         --quiet   # Claude API client
!pip install langgraph         --quiet   # multi-agent graph framework
# !pip install duckduckgo-search --quiet   # free web search (no API key needed)
!pip install ddgs              --quiet
!pip install requests          --quiet   # HTTP client for calling hosted REST APIs
!pip install python-dotenv     --quiet   # load .env files

print()
print("All packages installed successfully.")
print()
print("Package roles:")
print("  anthropic          → talk to Claude (the LLM powering every agent)")
print("  langgraph          → wire agents together as a directed state machine")
print("  duckduckgo-search  → free web search tool (you call it, someone else hosts it)")
print("  requests           → call any hosted REST API as an agent tool (Wikipedia, etc.)")
print("  python-dotenv      → securely load ANTHROPIC_API_KEY from .env")

/home/administrator/miniconda3/envs/bootcamp/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=3330942) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()
/home/administrator/miniconda3/envs/bootcamp/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=3330942) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()
/home/administrator/miniconda3/envs/bootcamp/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=3330942) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()
/home/administrator/miniconda3/envs/bootcamp/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=3330942) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()
/home/administrator/miniconda3/envs/bootcamp/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=3330942) is multi-threaded, use


All packages installed successfully.

Package roles:
  anthropic          → talk to Claude (the LLM powering every agent)
  langgraph          → wire agents together as a directed state machine
  duckduckgo-search  → free web search tool (you call it, someone else hosts it)
  requests           → call any hosted REST API as an agent tool (Wikipedia, etc.)
  python-dotenv      → securely load ANTHROPIC_API_KEY from .env


In [ ]:
# # For ruuning in colab 

# import os
# import getpass

# # Try Colab secrets first (recommended), then fall back to manual input
# try:
#     from google.colab import userdata
#     os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
#     print("API key loaded from Colab secrets.")
# except Exception:
#     # Running outside Colab — prompt for key
#     key = getpass.getpass("Paste your ANTHROPIC_API_KEY (input is hidden): ")
#     os.environ["ANTHROPIC_API_KEY"] = key
#     print("API key set.")

# import anthropic
# client = anthropic.Anthropic()
# MODEL  = "claude-haiku-4-5"   # fast + cheap — perfect for learning
# print(f"Client ready. Model: {MODEL}")

In [2]:
import os, json, re, math, operator, datetime, time
from pathlib import Path
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv
import requests
import anthropic

def find_dotenv_path():
    candidates = [
        Path.cwd() / ".env",
        Path.cwd().parent / ".env",
        Path.cwd().parent.parent / ".env",
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

dotenv_path = find_dotenv_path()
if dotenv_path:
    load_dotenv(dotenv_path=dotenv_path)
    print(f"Loaded .env from: {dotenv_path}")
else:
    print("Warning: no .env found — set ANTHROPIC_API_KEY manually.")

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise OSError("ANTHROPIC_API_KEY not found. Add it to your .env file.")

client     = anthropic.Anthropic()
MODEL_FAST = "claude-haiku-4-5"    # fast + cheap
MODEL_GOOD = "claude-sonnet-4-6"   # smarter, slower
print(f"Client ready. Fast model: {MODEL_FAST} | Smart model: {MODEL_GOOD}")

Loaded .env from: /home/administrator/Desktop/Bootcamp/.env
Client ready. Fast model: claude-haiku-4-5 | Smart model: claude-sonnet-4-6


---
# Part 1 — ReAct Recap: The Loop That Gives Agents Hands

## Day 2 in 60 seconds

On Day 2 you gave the agent **tools**. Instead of answering from memory alone, the agent can now:

```
while not done:
    THOUGHT     → agent reasons about what to do next
    ACTION      → agent calls a tool  (web_search, calculator, read_file)
    OBSERVATION → tool returns a result
    ↑ loop back with the new observation
FINAL ANSWER  → agent concludes when it has enough information
```

This is the **ReAct pattern** (Reason + Act, interleaved).

### Why it matters for today
On Day 3 the Researcher agent *is* a ReAct agent — it runs this loop inside one node of the multi-agent graph. The multi-agent architecture **wraps** the ReAct loop rather than replacing it.

```
Day 1: one LLM call
Day 2: one ReAct loop  (LLM ↔ tools, iteratively)
Day 3: multiple agents, each of which can run a ReAct loop
```

![gRPC Diagram](https://purelogics.com/wp-content/uploads/2025/03/ReAct-Agents-vs-Function-Calling-Agents-1.jpg)

In [9]:
# ── Tool implementations (carried over from Day 2) ─────────────────────────────

def web_search(query: str, max_results: int = 3) -> str:
    try:
        from ddgs import DDGS

        results = []
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=max_results):
                results.append(f"• {r['title']}: {r['body'][:200]}")

        return "\n".join(results) if results else "No results found."

    except ImportError:
        return "web_search unavailable: pip install ddgs"
    except Exception as e:
        return f"Search error: {e}"


def calculator(expression: str) -> str:
    safe_chars = set("0123456789+-*/().% ")
    if not all(c in safe_chars for c in expression):
        return "Error: only basic arithmetic"
    try:
        result = eval(expression, {"__builtins__": {}},  # noqa: S307
                      {"sqrt": math.sqrt, "pi": math.pi, "abs": abs})
        return str(result)
    except Exception as e:
        return f"Calculation error: {e}"


TOOL_REGISTRY = {"web_search": web_search, "calculator": calculator}

REACT_TOOLS = [
    {
        "name": "web_search",
        "description": "Search the web for current information on a topic.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string", "description": "Search query"}},
            "required": ["query"],
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a math expression. Use for ALL arithmetic.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string", "description": "e.g. '12 * 8.5'"}},
            "required": ["expression"],
        },
    },
]

print("Tools registered:", list(TOOL_REGISTRY.keys()))

Tools registered: ['web_search', 'calculator']


In [10]:
# ── ReAct loop function (reusable — we'll plug this into LangGraph later) ───────

def react_loop(question: str, system: str = None, max_iterations: int = 6) -> str:
    """
    Run a single ReAct loop.
    Returns the final text answer.
    """
    default_system = (
        "You are a research assistant with access to tools. "
        "Use web_search for factual questions. Use calculator for arithmetic. "
        "Reason before each tool call. Stop when you have a complete answer."
    )
    messages   = [{"role": "user", "content": question}]
    iteration  = 0

    print(f"\n{'─'*60}")
    print(f"  ReAct loop started | max {max_iterations} iterations")
    print(f"  Question: {question}")
    print(f"{'─'*60}")

    while iteration < max_iterations:
        iteration += 1
        print(f"\n  [Iteration {iteration}]")

        response = client.messages.create(
            model=MODEL_FAST,
            max_tokens=500,
            system=system or default_system,
            tools=REACT_TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        for block in response.content:
            if hasattr(block, "text") and block.text:
                print(f"  Thought: {block.text[:150]}")

        if response.stop_reason == "end_turn":
            final = next((b.text for b in response.content if hasattr(b, "text")), "")
            print(f"  → Done (end_turn)")
            return final

        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            fn  = TOOL_REGISTRY.get(block.name)
            obs = fn(**block.input) if fn else f"Unknown tool: {block.name}"
            print(f"  Action:      {block.name}({block.input})")
            print(f"  Observation: {obs[:120]}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": obs})

        messages.append({"role": "user", "content": tool_results})

    return "Max iterations reached."


# Run a quick ReAct demo
answer = react_loop("What is the square root of 144 multiplied by 7, and what is machine learning?")
print("\n" + "=" * 60)
print("  FINAL ANSWER")
print("=" * 60)
print(answer)


────────────────────────────────────────────────────────────
  ReAct loop started | max 6 iterations
  Question: What is the square root of 144 multiplied by 7, and what is machine learning?
────────────────────────────────────────────────────────────

  [Iteration 1]
  Thought: I'll help you with both of these questions. Let me calculate the math problem and search for information about machine learning.
  Action:      calculator({'expression': 'sqrt(144) * 7'})
  Observation: Error: only basic arithmetic
  Action:      web_search({'query': 'what is machine learning'})
  Observation: • Machine learning - Wikipedia: Machine learning (ML) is a field of study in artificial intelligence concerned with the 

  [Iteration 2]
  Thought: Let me recalculate that math problem using basic arithmetic:
  Action:      calculator({'expression': '12 * 7'})
  Observation: 84

  [Iteration 3]
  Thought: **Math Problem:**
The square root of 144 is 12, and 12 multiplied by 7 equals **84**.

**Machine Le

### Key takeaway
The `react_loop()` function above is a **self-contained capability**. In the multi-agent system we're about to build, the Researcher node will call this function. This is how Day 2 and Day 3 connect — the Researcher is just the ReAct agent embedded inside a larger graph.

```
Multi-agent graph (Day 3)
  └── Researcher node
        └── react_loop()  ← the Day 2 code, unchanged
```

---
# Part 1b — Using Hosted Tools: You Don't Need to Build Everything

## The key insight

When we defined `web_search` and `calculator` on Day 2, we **wrote the functions ourselves**.
But in a real agentic system, most tools are things that already exist on the internet as hosted services:

```
Build it yourself        vs.     Use a hosted service
─────────────────────────────────────────────────────
def web_search(q):               DuckDuckGo API  ✓ already hosted
    # scrape HTML...             Wikipedia REST  ✓ already hosted
    # parse results...           WolframAlpha    ✓ already hosted
    # handle errors...           OpenWeatherMap  ✓ already hosted
    ...                          Tavily Search   ✓ already hosted
```

**Your job as an agent developer is to write the thin adapter** — a Python function that calls the hosted service and formats the result for the LLM. The heavy lifting (crawling, indexing, computation) is someone else's infrastructure.

## Wikipedia REST API — a real hosted tool

Wikipedia exposes a free, no-API-key REST endpoint:

```
GET https://en.wikipedia.org/api/rest_v1/page/summary/{page_title}
→ Returns: title, extract (plain English summary), URL
```

We'll wrap it in under 10 lines and plug it into our existing ReAct loop — zero changes to the loop itself."

![gRPC Diagram](https://framerusercontent.com/images/gaNIC7XU7Koj4LYCdEqQ1aVBS0.jpeg)

In [6]:
# ── Hosted Tool 1: Wikipedia REST API ─────────────────────────────────────────
# This is a REAL live API call to Wikipedia's servers.
# No API key. No billing. No building a crawler.
# We just wrap the response into the format our agent expects.

import requests

def wikipedia_summary(topic: str) -> str:
    url = f"https://en.wikipedia.org/api/rest_v1/page/summary/{topic.replace(' ', '_')}"
    
    headers = {
        "Accept": "application/json",
        "User-Agent": "MyResearchAgent/1.0 (your_email@example.com)"  # REQUIRED
    }

    try:
        resp = requests.get(url, headers=headers, timeout=5)
        
        if resp.status_code == 200:
            data = resp.json()
            title   = data.get("title", topic)
            extract = data.get("extract", "No summary available.")
            page_url = data.get("content_urls", {}).get("desktop", {}).get("page", "")
            return f"Wikipedia — {title}:\n{extract}\nSource: {page_url}"

        if resp.status_code == 404:
            return f"Wikipedia: no page found for '{topic}'."

        return f"Wikipedia error: HTTP {resp.status_code} - {resp.text}"

    except requests.exceptions.Timeout:
        return "Wikipedia: request timed out."
    except Exception as e:
        return f"Wikipedia error: {e}"


# ── Quick test — call the real Wikipedia API right now ─────────────────────────
print("Calling Wikipedia REST API (live)...\n")
result = wikipedia_summary("Machine learning")
print(result[:500])
print("\n" + "─" * 60)
print("Notice: we made a real HTTP request to en.wikipedia.org.")
print("Wikipedia's servers did all the work — we just formatted the response.")

Calling Wikipedia REST API (live)...

Wikipedia — Machine learning:
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalize to unseen data, and thus perform tasks without being explicitly programmed. Advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
Source: https://en.wikipedia.org/wiki/Machine

────────────────────────────────────────────────────────────
Notice: we made a real HTTP request to en.wikipedia.org.
Wikipedia's servers did all the work — we just formatted the response.


In [11]:
# ── Plug the Wikipedia tool into the existing ReAct loop ──────────────────────
# We add ONE entry to TOOL_REGISTRY and ONE tool schema.
# The react_loop() function from Part 1 needs zero changes.

TOOL_REGISTRY["wikipedia"] = lambda topic: wikipedia_summary(topic)

REACT_TOOLS.append({
    "name": "wikipedia",
    "description": (
        "Fetch a reliable summary of a topic from Wikipedia. "
        "Use this for factual background on well-known concepts, people, or events. "
        "Prefer this over web_search when you need an encyclopaedic overview."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "topic": {
                "type": "string",
                "description": "Wikipedia page title, e.g. 'Neural network' or 'World War II'",
            }
        },
        "required": ["topic"],
    },
})

print("Tools now available to the agent:", list(TOOL_REGISTRY.keys()))
print()
print("─" * 60)
print("  The agent can now CHOOSE between:")
print("  • web_search   → DuckDuckGo (hosted by DuckDuckGo)")
print("  • wikipedia    → Wikipedia  (hosted by Wikimedia Foundation)")
print("  • calculator   → safe eval  (runs locally)")
print("─" * 60)

# Run the agent — it will decide when to use Wikipedia vs web_search
print()
answer = react_loop(
    "What is the difference between supervised and unsupervised learning? "
    "Also, what year was the first ImageNet competition held?",
    max_iterations=5,
)
print("\n" + "═" * 60)
print("  AGENT ANSWER (with hosted Wikipedia + DuckDuckGo tools)")
print("═" * 60)
print(answer)

Tools now available to the agent: ['web_search', 'calculator', 'wikipedia']

────────────────────────────────────────────────────────────
  The agent can now CHOOSE between:
  • web_search   → DuckDuckGo (hosted by DuckDuckGo)
  • wikipedia    → Wikipedia  (hosted by Wikimedia Foundation)
  • calculator   → safe eval  (runs locally)
────────────────────────────────────────────────────────────


────────────────────────────────────────────────────────────
  ReAct loop started | max 5 iterations
  Question: What is the difference between supervised and unsupervised learning? Also, what year was the first ImageNet competition held?
────────────────────────────────────────────────────────────

  [Iteration 1]
  Thought: I'll search for information about the ImageNet competition timing, and I can explain the difference between supervised and unsupervised learning from 
  Action:      web_search({'query': 'first ImageNet competition year'})
  Observation: • ImageNet - Wikipedia: The ImageNet

### What just happened

The agent chose **which hosted service to call** based on the question:
- Background on ML concepts → `wikipedia` (encyclopaedic, stable)
- Specific event date like ImageNet → `web_search` (DuckDuckGo)

```
Agent decides: "I need facts about supervised learning"
    → calls wikipedia("Supervised learning")
    → Wikipedia's servers respond instantly
    → Agent reads the result and continues

Agent decides: "I need the ImageNet competition year"
    → calls web_search("first ImageNet competition year")
    → DuckDuckGo's servers search the web
    → Agent reads the result and concludes
```

**You didn't crawl Wikipedia. You didn't build a search engine.** You wrote ~10 lines wrapping two free hosted services, and the agent uses them intelligently.

### Exercise 1b — Add a third hosted service
The [Open Library API](https://openlibrary.org/developers/api) returns book metadata — title, author, year, subject — for any ISBN or search term. It's free and requires no API key.

```python
def open_library_search(query: str) -> str:
    resp = requests.get(
        "https://openlibrary.org/search.json",
        params={"q": query, "limit": 3},
        timeout=5
    )
    docs = resp.json().get("docs", [])
    return "\n".join(
        f"• {d.get('title')} by {', '.join(d.get('author_name', ['Unknown']))} ({d.get('first_publish_year', '?')})"
        for d in docs
    )
```

Add it to `TOOL_REGISTRY` and `REACT_TOOLS`, then ask the agent: *"Recommend 3 foundational books on machine learning."*

---
# Part 2 — Why Multi-Agent? The Specialisation Argument

## The problem with one giant agent

Imagine asking one person to simultaneously:
- Manage the project
- Research all the facts
- Write the report
- Edit the report

They'd be context-switching constantly, doing each task poorly. The same is true for LLMs.

## The four reasons to split

| Reason | Explanation |
|--------|-------------|
| **Specialisation** | Each agent has a focused system prompt → better at its one job |
| **Cost control** | Use a cheap fast model (Haiku) for routing; expensive model (Sonnet) only where it matters |
| **Debuggability** | You can run any node in isolation and inspect its output |
| **Parallelism** | Independent agents can run concurrently (Day 4 adds this) |

## A concrete comparison

We'll send the same question to:
1. A **single monolithic agent** (like Day 1)
2. A **two-agent pipeline** (Planner → Writer)

Watch the output quality difference.

In [12]:
QUESTION = "Explain the causes and economic effects of the 2008 financial crisis for a first-year economics student."

# ── Approach 1: monolithic agent (one big call) ────────────────────────────────
print("=" * 65)
print("  APPROACH 1: Monolithic Agent (single LLM call)")
print("=" * 65)

mono_response = client.messages.create(
    model=MODEL_FAST,
    max_tokens=600,
    system="You are a helpful economics tutor. Answer clearly.",
    messages=[{"role": "user", "content": QUESTION}],
)
mono_answer = mono_response.content[0].text
print(mono_answer[:500], "..." if len(mono_answer) > 500 else "")
print(f"\n  [Tokens used: {mono_response.usage.output_tokens}]")

print()

# ── Approach 2: two-agent pipeline ────────────────────────────────────────────
print("=" * 65)
print("  APPROACH 2: Two-Agent Pipeline (Planner → Writer)")
print("=" * 65)

# Agent 1: Planner
print("\n  [Planner] Creating subtasks...")
plan_response = client.messages.create(
    model=MODEL_FAST,
    max_tokens=200,
    system=(
        "You are a research planner. "
        "Given a question, produce a numbered list of 4 focused subtopics to cover. "
        "Be specific. One line per subtopic."
    ),
    messages=[{"role": "user", "content": QUESTION}],
)
plan = plan_response.content[0].text
print(plan)

# Agent 2: Writer (receives the plan)
print("\n  [Writer] Drafting answer using the plan...")
writer_response = client.messages.create(
    model=MODEL_GOOD,
    max_tokens=700,
    system=(
        "You are an expert economics tutor writing for first-year university students. "
        "Follow the research plan exactly. "
        "Use clear language, analogies, and real examples. "
        "Structure your answer with section headings."
    ),
    messages=[{
        "role": "user",
        "content": f"Research plan:\n{plan}\n\nQuestion to answer:\n{QUESTION}",
    }],
)
pipeline_answer = writer_response.content[0].text
print(pipeline_answer[:600], "..." if len(pipeline_answer) > 600 else "")
total_tokens = plan_response.usage.output_tokens + writer_response.usage.output_tokens
print(f"\n  [Tokens used: {total_tokens} across 2 agents]")

print()
print("=" * 65)
print("  NOTICE: The pipeline answer is more structured because")
print("  the Writer received a specific research plan — it knew")
print("  exactly what sections to write and in what order.")
print("=" * 65)

  APPROACH 1: Monolithic Agent (single LLM call)
# The 2008 Financial Crisis: Causes and Effects

## Main Causes

**Housing Bubble**
- Banks issued subprime mortgages to people with poor credit and limited ability to repay
- Housing prices soared as demand increased, creating unsustainable speculation
- Banks stopped checking borrowers' creditworthiness, assuming prices would keep rising

**Complex Financial Instruments**
- Banks bundled mortgages into securities (mortgage-backed securities) and sold them worldwide
- These were so complex that ...

  [Tokens used: 391]

  APPROACH 2: Two-Agent Pipeline (Planner → Writer)

  [Planner] Creating subtasks...
# Research Plan: 2008 Financial Crisis

1. **Housing bubble and subprime mortgage lending** – How banks issued mortgages to unqualified borrowers, inflated home prices beyond real value, and created unsustainable debt obligations.

2. **Securitization and credit rating failures** – How risky mortgages were packaged into complex financi

---
# Part 3 — LangGraph Basics: State, Nodes, and Edges

## The mental model

Think of LangGraph as an assembly line where a **shared notepad** travels down the line.
Each worker (node) reads the notepad, does their job, and writes their results back.

```
Shared Notepad (State Dict)
┌─────────────────────────────────┐
│ question:   "Explain inflation" │
│ plan:       ""  ← Planner fills │
│ answer:     ""  ← Writer fills  │
│ agent_log:  []  ← everyone adds │
└─────────────────────────────────┘
       ↓              ↓             ↓
   [Planner]     [Writer]      [Critic]
    reads:         reads:        reads:
    question       question      answer
    writes:        plan          writes:
    plan           writes:       score
                   answer
```

### Three rules
1. **Each node reads only what it needs** from the state
2. **Each node writes only the fields it owns** — never overwrite another agent's work
3. **Edges decide the order** — who runs next (can be conditional)

Let's build the simplest possible graph: two nodes, one path.

![gRPC Diagram](https://miro.medium.com/v2/resize:fit:1200/1*iJ5U8OzJRNFrvt3UuL4tuQ.png)

In [13]:
from langgraph.graph import StateGraph, START, END
print("LangGraph imported successfully.")

LangGraph imported successfully.


In [14]:
# ── STEP 1: Define the State ───────────────────────────────────────────────────
# TypedDict = a regular dict but with type hints.
# LangGraph uses this to know what fields exist and their types.

class SimpleState(TypedDict):
    question  : str        # INPUT  — never changes
    plan      : str        # set by Planner node
    answer    : str        # set by Writer node
    agent_log : list[str]  # everyone appends here


# ── STEP 2: Define nodes ───────────────────────────────────────────────────────
# Each node is a plain Python function.
# Input:  the current state (TypedDict)
# Output: a dict of ONLY the fields this node wants to update

def planner_node(state: SimpleState) -> dict:
    print("  [Planner] Running...")
    response = client.messages.create(
        model=MODEL_FAST,
        max_tokens=200,
        system="List 3 numbered steps to answer this question. Be concise.",
        messages=[{"role": "user", "content": state["question"]}],
    )
    plan = response.content[0].text
    print(f"  [Planner] Done. Steps: {plan.splitlines()[0]}...")
    return {
        "plan"     : plan,
        "agent_log": state["agent_log"] + ["Planner completed"],
    }


def writer_node(state: SimpleState) -> dict:
    print("  [Writer] Running...")
    response = client.messages.create(
        model=MODEL_GOOD,
        max_tokens=500,
        system="You are a university writing assistant. Follow the plan and answer clearly.",
        messages=[{
            "role": "user",
            "content": f"Plan:\n{state['plan']}\n\nQuestion:\n{state['question']}",
        }],
    )
    answer = response.content[0].text
    print(f"  [Writer] Done. Answer: {answer[:60]}...")
    return {
        "answer"   : answer,
        "agent_log": state["agent_log"] + ["Writer completed"],
    }


# ── STEP 3: Build the graph ────────────────────────────────────────────────────

def build_simple_graph():
    graph = StateGraph(SimpleState)

    graph.add_node("planner", planner_node)
    graph.add_node("writer",  writer_node)

    # Edges: START → planner → writer → END
    graph.add_edge(START,     "planner")
    graph.add_edge("planner", "writer")
    graph.add_edge("writer",  END)

    return graph.compile()


# ── STEP 4: Run it ─────────────────────────────────────────────────────────────
print("Building and running 2-node graph...\n")
app = build_simple_graph()

initial_state: SimpleState = {
    "question" : "What is photosynthesis and why is it important for life on Earth?",
    "plan"     : "",
    "answer"   : "",
    "agent_log": [],
}

result = app.invoke(initial_state)

print("\n" + "─" * 60)
print("  GRAPH RESULT")
print("─" * 60)
print(f"  Agent log: {result['agent_log']}")
print(f"\n  Plan used:")
print(result["plan"])
print(f"\n  Final answer:")
print(result["answer"])

Building and running 2-node graph...

  [Planner] Running...
  [Planner] Done. Steps: # Photosynthesis: Definition and Importance...
  [Writer] Running...
  [Writer] Done. Answer: # Photosynthesis: Definition and Importance

## What Is Phot...

────────────────────────────────────────────────────────────
  GRAPH RESULT
────────────────────────────────────────────────────────────
  Agent log: ['Planner completed', 'Writer completed']

  Plan used:
# Photosynthesis: Definition and Importance

## What It Is
1. **Process Definition**: Photosynthesis is a biochemical process where plants, algae, and some bacteria convert sunlight, water, and carbon dioxide into glucose (sugar) and oxygen using light energy.

2. **The Basic Equation**: Light + CO₂ + H₂O → Glucose + O₂

## Why It Matters for Life on Earth

3. **Critical Importance**:
   - **Oxygen Production** – Creates the oxygen we breathe
   - **Food Source** – Forms the base of most food chains; plants feed herbivores, which feed carnivor

### What just happened?

```
START
  │
  ▼
[planner_node]  reads: question          writes: plan, agent_log
  │
  ▼
[writer_node]   reads: question, plan    writes: answer, agent_log
  │
  ▼
END
```

The `result` dict contains the **final state** after all nodes ran. Every field that any node wrote is accessible there.

**Important:** Neither node knows the other exists. They only know about the state dict. You can swap the writer for a different writer without touching the planner.

---
# Part 4 — Conditional Routing: State Machines

## Static edges vs. conditional edges

So far our edges are **static**: planner → writer, always.

**Conditional edges** route differently based on the state. This is what makes LangGraph a *state machine*:

```python
# Static edge:
graph.add_edge("node_a", "node_b")          # always goes to node_b

# Conditional edge:
graph.add_conditional_edges("node_a", my_router_fn)  # router decides
```

## Use case: smart routing + revision loop

We'll build a graph where:
1. An **Orchestrator** classifies the question as `simple` or `complex`
2. Simple questions go to a **QuickAnswer** node (cheap, fast)
3. Complex questions go to a **DeepResearch** node (thorough)
4. All answers pass through a **Critic** node
5. If score < 7, the Critic sends the answer back to a **Revise** node → back to Critic (loop!)

```
START → Orchestrator →(simple)→ QuickAnswer ─┐
                    →(complex)→ DeepResearch ─┤
                                              ▼
                                           Critic →(score≥7)→ END
                                              ↑←(score<7)← Revise ←┘
```

In [15]:
class RoutedState(TypedDict):
    question      : str
    complexity    : str    # "simple" | "complex"
    answer        : str
    revision_count: int
    quality_score : int
    agent_log     : list[str]


# ── Nodes ──────────────────────────────────────────────────────────────────────

def orchestrator_node(state: RoutedState) -> dict:
    response = client.messages.create(
        model=MODEL_FAST,
        max_tokens=10,
        system=(
            "Classify this question as SIMPLE or COMPLEX.\n"
            "SIMPLE = answerable from general knowledge in 2-3 sentences.\n"
            "COMPLEX = requires research, math, or multi-step reasoning.\n"
            "Reply with ONE word only: SIMPLE or COMPLEX"
        ),
        messages=[{"role": "user", "content": state["question"]}],
    )
    complexity = response.content[0].text.strip().upper()
    if complexity not in ("SIMPLE", "COMPLEX"):
        complexity = "COMPLEX"
    print(f"  [Orchestrator] '{state['question'][:50]}...' → {complexity}")
    return {
        "complexity": complexity.lower(),
        "agent_log" : state["agent_log"] + [f"Orchestrator → {complexity}"],
    }


def quick_answer_node(state: RoutedState) -> dict:
    response = client.messages.create(
        model=MODEL_FAST, max_tokens=200,
        system="Answer this clearly and concisely in 2-3 sentences.",
        messages=[{"role": "user", "content": state["question"]}],
    )
    print("  [QuickAnswer] Answered directly.")
    return {
        "answer"       : response.content[0].text,
        "quality_score": 8,
        "agent_log"    : state["agent_log"] + ["QuickAnswer"],
    }


def deep_research_node(state: RoutedState) -> dict:
    response = client.messages.create(
        model=MODEL_GOOD, max_tokens=600,
        system=(
            "You are a thorough academic researcher. "
            "Provide a detailed, well-structured answer with examples and clear sections."
        ),
        messages=[{"role": "user", "content": state["question"]}],
    )
    print("  [DeepResearch] Produced detailed answer.")
    return {
        "answer"   : response.content[0].text,
        "agent_log": state["agent_log"] + ["DeepResearch"],
    }


def critic_node(state: RoutedState) -> dict:
    response = client.messages.create(
        model=MODEL_FAST, max_tokens=50,
        system="Score this answer 1-10. Reply with ONLY the integer.",
        messages=[{"role": "user", "content":
            f"Question: {state['question']}\nAnswer: {state['answer'][:300]}"}],
    )
    try:
        score = int(response.content[0].text.strip())
    except ValueError:
        score = 7
    print(f"  [Critic] Score: {score}/10 | Revisions so far: {state['revision_count']}")
    return {
        "quality_score": score,
        "agent_log"    : state["agent_log"] + [f"Critic(score={score})"],
    }


def revise_node(state: RoutedState) -> dict:
    response = client.messages.create(
        model=MODEL_GOOD, max_tokens=600,
        system="Improve this answer. Make it clearer, more accurate, and better structured.",
        messages=[{"role": "user", "content":
            f"Question: {state['question']}\nCurrent answer: {state['answer']}\nImprove it."}],
    )
    print("  [Revise] Improved the answer.")
    return {
        "answer"        : response.content[0].text,
        "revision_count": state["revision_count"] + 1,
        "agent_log"     : state["agent_log"] + ["Revise"],
    }


# ── Routing functions (these return node names, not values) ────────────────────

def route_by_complexity(state: RoutedState) -> Literal["quick_answer", "deep_research"]:
    return "quick_answer" if state["complexity"] == "simple" else "deep_research"


def route_after_critic(state: RoutedState) -> Literal["revise", "__end__"]:
    if state["quality_score"] < 7 and state["revision_count"] < 2:
        print(f"  [Router] Score {state['quality_score']} < 7 → revising")
        return "revise"
    print(f"  [Router] Score {state['quality_score']} → done")
    return END


# ── Build the graph ────────────────────────────────────────────────────────────

def build_routed_graph():
    g = StateGraph(RoutedState)
    g.add_node("orchestrator", orchestrator_node)
    g.add_node("quick_answer", quick_answer_node)
    g.add_node("deep_research", deep_research_node)
    g.add_node("critic",       critic_node)
    g.add_node("revise",       revise_node)

    g.add_edge(START, "orchestrator")
    g.add_conditional_edges("orchestrator", route_by_complexity)
    g.add_edge("quick_answer",  "critic")
    g.add_edge("deep_research", "critic")
    g.add_conditional_edges("critic", route_after_critic)
    g.add_edge("revise", "critic")

    return g.compile()


routed_app = build_routed_graph()
print("Routed graph built successfully.")

Routed graph built successfully.


In [16]:
# Test with a SIMPLE question
print("=" * 60)
print("  TEST 1: Simple question")
print("=" * 60)

result_simple = routed_app.invoke({
    "question"      : "What is the capital of France?",
    "complexity"    : "",
    "answer"        : "",
    "revision_count": 0,
    "quality_score" : 0,
    "agent_log"     : [],
})

print(f"\n  Path: {' → '.join(result_simple['agent_log'])}")
print(f"  Score: {result_simple['quality_score']}/10")
print(f"  Answer: {result_simple['answer'][:200]}")

  TEST 1: Simple question
  [Orchestrator] 'What is the capital of France?...' → SIMPLE
  [QuickAnswer] Answered directly.
  [Critic] Score: 10/10 | Revisions so far: 0
  [Router] Score 10 → done

  Path: Orchestrator → SIMPLE → QuickAnswer → Critic(score=10)
  Score: 10/10
  Answer: The capital of France is Paris. It is located in the north-central part of the country along the Seine River and is the country's largest city.


In [17]:
# Test with a COMPLEX question
print("=" * 60)
print("  TEST 2: Complex question")
print("=" * 60)

result_complex = routed_app.invoke({
    "question"      : "Explain how CRISPR-Cas9 gene editing works and its ethical implications for medicine.",
    "complexity"    : "",
    "answer"        : "",
    "revision_count": 0,
    "quality_score" : 0,
    "agent_log"     : [],
})

print(f"\n  Path: {' → '.join(result_complex['agent_log'])}")
print(f"  Score: {result_complex['quality_score']}/10")
print(f"  Revisions: {result_complex['revision_count']}")
print(f"\n  Answer (first 400 chars):")
print(result_complex["answer"][:400])

  TEST 2: Complex question
  [Orchestrator] 'Explain how CRISPR-Cas9 gene editing works and its...' → COMPLEX
  [DeepResearch] Produced detailed answer.
  [Critic] Score: 8/10 | Revisions so far: 0
  [Router] Score 8 → done

  Path: Orchestrator → COMPLEX → DeepResearch → Critic(score=8)
  Score: 8/10
  Revisions: 0

  Answer (first 400 chars):
# CRISPR-Cas9 Gene Editing: Mechanisms and Ethical Implications

---

## Part I: The Science — How CRISPR-Cas9 Works

### Historical Background

CRISPR (**C**lustered **R**egularly **I**nterspaced **S**hort **P**alindromic **R**epeats) was first identified in 1987 by Japanese microbiologist Yoshizumi Ishino, who noticed unusual repeated sequences in bacterial DNA. The biological function — an adap


### Exercise 4.1 — Force a revision

Change the critic threshold from 7 to 9 and observe the loop in action. Edit `route_after_critic` above:

```python
if state["quality_score"] < 9 and state["revision_count"] < 2:  # changed 7 → 9
```

Rebuild the graph and rerun the complex question. You should see the Revise node activate.

**Key insight:** The routing logic lives *outside* the agents. You can tighten or loosen quality requirements without touching any agent's code.

---
# Part 5 — Agent Handoffs: How Agents Pass Context

## The handoff contract

A handoff is clean when:
- **Agent A** writes exactly one section of the state
- **Agent B** reads only what it needs — not the whole history
- Neither agent modifies the other's field after the fact

This is like a relay race: you hand the baton cleanly, you don't chase the next runner.

## The `Annotated` trick for lists

When two agents both try to update `agent_log`, the second one overwrites the first.
To fix this, use `Annotated[list, operator.add]` — LangGraph will **append** instead of replace:

```python
# Without Annotated → second node overwrites first node's log
agent_log: list[str]

# With Annotated → logs are accumulated across all nodes
agent_log: Annotated[list[str], operator.add]
```

## Full 4-agent pipeline with handoffs

```
START → Planner → Researcher → Writer → Critic → END

Planner    reads: question
           writes: subtasks

Researcher reads: question, subtasks
           writes: research (dict of findings per subtask)

Writer     reads: question, research
           writes: draft

Critic     reads: question, draft
           writes: final_answer
```

In [18]:
class PipelineState(TypedDict):
    question     : str
    subtasks     : list[str]                              # Planner writes
    research     : dict[str, str]                         # Researcher writes
    draft        : str                                    # Writer writes
    final_answer : str                                    # Critic writes
    handoff_log  : Annotated[list[str], operator.add]     # all agents append


def _log_entry(agent: str, msg: str) -> list[str]:
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    entry = f"[{ts}] [{agent}] {msg}"
    print(f"  {entry}")
    return [entry]


def pipeline_planner(state: PipelineState) -> dict:
    response = client.messages.create(
        model=MODEL_FAST,
        max_tokens=200,
        system=(
            'You are a task planner. Output a JSON array of 3 research subtasks. '
            'Example: ["Find X", "Explain Y", "Compare Z"]. Output ONLY the JSON array.'
        ),
        messages=[{"role": "user", "content": state["question"]}],
    )
    text  = response.content[0].text
    match = re.search(r"\[.*\]", text, re.DOTALL)
    try:
        subtasks = json.loads(match.group()) if match else ["Research main topic", "Find examples", "Summarise"]
    except Exception:
        subtasks = ["Research main topic", "Find examples", "Summarise"]
    return {
        "subtasks"   : subtasks,
        "handoff_log": _log_entry("Planner", f"Created {len(subtasks)} subtasks"),
    }


def pipeline_researcher(state: PipelineState) -> dict:
    findings: dict[str, str] = {}
    for subtask in state["subtasks"]:
        response = client.messages.create(
            model=MODEL_FAST, max_tokens=200,
            system="Answer this research subtask in 2-3 concise sentences. Be factual.",
            messages=[{"role": "user", "content":
                f"Subtask: {subtask}\nContext: {state['question']}"}],
        )
        findings[subtask] = response.content[0].text
    return {
        "research"   : findings,
        "handoff_log": _log_entry("Researcher", f"Completed {len(findings)} findings"),
    }


def pipeline_writer(state: PipelineState) -> dict:
    research_block = "\n\n".join(
        f"**{task}**\n{finding}"
        for task, finding in state["research"].items()
    )
    response = client.messages.create(
        model=MODEL_GOOD, max_tokens=700,
        system=(
            "You are an academic writer. Synthesise the research notes into a clear, "
            "well-structured answer. Intro → main points → conclusion. "
            "Write for a first-year student."
        ),
        messages=[{"role": "user", "content":
            f"Question: {state['question']}\n\nResearch notes:\n{research_block}"}],
    )
    return {
        "draft"      : response.content[0].text,
        "handoff_log": _log_entry("Writer", "Draft completed"),
    }


def pipeline_critic(state: PipelineState) -> dict:
    response = client.messages.create(
        model=MODEL_FAST, max_tokens=500,
        system=(
            "You are a meticulous academic editor. "
            "Improve this draft: fix clarity, remove redundancy, ensure claims are supported. "
            "Return the improved version only."
        ),
        messages=[{"role": "user", "content":
            f"Question: {state['question']}\n\nDraft:\n{state['draft']}"}],
    )
    return {
        "final_answer": response.content[0].text,
        "handoff_log" : _log_entry("Critic", "Final answer approved"),
    }


def build_pipeline_graph():
    g = StateGraph(PipelineState)
    g.add_node("planner",    pipeline_planner)
    g.add_node("researcher", pipeline_researcher)
    g.add_node("writer",     pipeline_writer)
    g.add_node("critic",     pipeline_critic)

    g.add_edge(START,        "planner")
    g.add_edge("planner",    "researcher")
    g.add_edge("researcher", "writer")
    g.add_edge("writer",     "critic")
    g.add_edge("critic",     END)

    return g.compile()


pipeline_app = build_pipeline_graph()

print("\n" + "=" * 60)
print("  Running 4-agent pipeline...")
print("=" * 60)

pipeline_result = pipeline_app.invoke({
    "question"    : "How does the immune system fight viral infections?",
    "subtasks"    : [],
    "research"    : {},
    "draft"       : "",
    "final_answer": "",
    "handoff_log" : [],
})

print("\n" + "─" * 60)
print("  HANDOFF LOG")
print("─" * 60)
for entry in pipeline_result["handoff_log"]:
    print(f"  {entry}")

print("\n" + "─" * 60)
print("  SUBTASKS the Planner created:")
print("─" * 60)
for i, t in enumerate(pipeline_result["subtasks"], 1):
    print(f"  {i}. {t}")

print("\n" + "─" * 60)
print("  FINAL ANSWER")
print("─" * 60)
print(pipeline_result["final_answer"])


  Running 4-agent pipeline...
  [06:26:52] [Planner] Created 3 subtasks
  [06:27:00] [Researcher] Completed 3 findings
  [06:27:14] [Writer] Draft completed
  [06:27:17] [Critic] Final answer approved

────────────────────────────────────────────────────────────
  HANDOFF LOG
────────────────────────────────────────────────────────────
  [06:26:52] [Planner] Created 3 subtasks
  [06:27:00] [Researcher] Completed 3 findings
  [06:27:14] [Writer] Draft completed
  [06:27:17] [Critic] Final answer approved

────────────────────────────────────────────────────────────
  SUBTASKS the Planner created:
────────────────────────────────────────────────────────────
  1. Explain the role of innate immunity in detecting and responding to viral infections
  2. Describe how adaptive immunity produces antibodies and T-cells to eliminate viruses
  3. Compare the mechanisms of viral clearance by interferon signaling and cytotoxic immune responses

──────────────────────────────────────────────────────

### Exercise 5.1 — Inspect what each agent actually received

Add a `print(state)` at the start of `pipeline_researcher` to see exactly what the Researcher receives.
Notice it gets `question` and `subtasks` — but NOT `draft` or `final_answer` (those don't exist yet).

This is the handoff contract in action: each agent only sees the state as it exists *at the moment it runs*.

---
# Part 6 — Lab: Full 5-Agent Platform

## The complete system

Now we combine everything into a production-quality 5-agent graph:

| Agent | Model | Responsibility | Reads | Writes |
|-------|-------|----------------|-------|--------|
| **Orchestrator** | Haiku (fast) | Classify question → choose route | question | route |
| **Planner** | Sonnet (smart) | Break into subtasks | question | subtasks |
| **Researcher** | Sonnet | Gather findings per subtask | question, subtasks | research |
| **Writer** | Sonnet | Synthesise draft | question, research, critique | draft |
| **Critic** | Haiku | Score + approve or loop back | question, draft | critique, final_answer, quality_score |

```
START
  │
  ▼
Orchestrator ──────────────────────────────────┐
  │ (route = "research" | "plan" | "summarize") │
  ▼                                            │
Planner                                        │
  │                                            │
  ▼                                            │
Researcher                                     │
  │                                            │
  ▼                                            │
Writer ◄───────────────────────────────────────┤
  │                                (revision loop)
  ▼                                            │
Critic ──(score ≥ 7)──► END                    │
       ──(score < 7)──► Revise ────────────────┘
```

In [ ]:
# ── Full State ─────────────────────────────────────────────────────────────────

class AssignmentState(TypedDict):
    # Input
    question      : str

    # Orchestrator
    route         : str                                  # research | plan | summarize

    # Planner
    subtasks      : list[str]

    # Researcher
    research      : dict[str, str]

    # Writer
    draft         : str

    # Critic
    critique      : str
    final_answer  : str
    quality_score : int

    # Meta
    revision_count: int
    agent_log     : Annotated[list[str], operator.add]
    started_at    : str


def _call(model: str, system: str, user: str, max_tokens: int = 600) -> str:
    response = client.messages.create(
        model=model, max_tokens=max_tokens, system=system,
        messages=[{"role": "user", "content": user}],
    )
    return response.content[0].text


def _log(agent: str, msg: str) -> list[str]:
    ts    = datetime.datetime.now().strftime("%H:%M:%S")
    entry = f"[{ts}] [{agent}] {msg}"
    print(f"  {entry}")
    return [entry]


print("State and helpers defined.")

In [ ]:
# ── Agent 1: Orchestrator ──────────────────────────────────────────────────────

def orchestrator(state: AssignmentState) -> dict:
    result = _call(
        MODEL_FAST,
        system=(
            "You are a routing agent. Classify the question into ONE route:\n"
            "  research  — needs information gathering\n"
            "  plan      — needs task breakdown or project planning\n"
            "  summarize — needs a summary of a topic\n"
            "Reply with ONLY one word: research, plan, or summarize"
        ),
        user=state["question"],
        max_tokens=10,
    )
    route = result.strip().lower()
    if route not in ("research", "plan", "summarize"):
        route = "research"
    return {
        "route"    : route,
        "agent_log": _log("Orchestrator", f"Route: {route}"),
    }


# ── Agent 2: Planner ───────────────────────────────────────────────────────────

def planner(state: AssignmentState) -> dict:
    result = _call(
        MODEL_GOOD,
        system=(
            'You are a research planner. Output a JSON array of 3-4 specific subtasks. '
            'Output ONLY the JSON array. Example: ["Find X", "Explain Y", "Compare Z"]'
        ),
        user=state["question"],
        max_tokens=200,
    )
    match = re.search(r"\[.*\]", result, re.DOTALL)
    try:
        subtasks = json.loads(match.group()) if match else ["Research topic", "Find examples", "Identify key concepts"]
    except Exception:
        subtasks = ["Research topic", "Find examples", "Identify key concepts"]
    return {
        "subtasks" : subtasks,
        "agent_log": _log("Planner", f"{len(subtasks)} subtasks: {subtasks}"),
    }


# ── Agent 3: Researcher (wraps the ReAct loop for real tool use) ───────────────

def researcher(state: AssignmentState) -> dict:
    findings: dict[str, str] = {}
    for subtask in state["subtasks"]:
        # Each subtask runs its own mini ReAct loop (from Part 1)
        finding = react_loop(
            question=f"Research subtask: {subtask}\nContext: {state['question']}",
            system=(
                "You are a precise research assistant. "
                "Use web_search for factual information. "
                "Summarise your findings in 3-4 sentences."
            ),
            max_iterations=3,
        )
        findings[subtask] = finding
    return {
        "research" : findings,
        "agent_log": _log("Researcher", f"Gathered {len(findings)} findings via ReAct"),
    }


# ── Agent 4: Writer ────────────────────────────────────────────────────────────

def writer(state: AssignmentState) -> dict:
    research_block = "\n\n".join(
        f"• {task}:\n  {finding}"
        for task, finding in state["research"].items()
    )
    # If this is a revision, include the critique
    critique_context = ""
    if state.get("critique") and state["revision_count"] > 0:
        critique_context = f"\n\nPrevious critique to address:\n{state['critique']}"

    draft = _call(
        MODEL_GOOD,
        system=(
            "You are an expert academic writer for university students.\n"
            "Synthesise the research notes into a clear, coherent answer.\n"
            "Structure: brief intro → main points → conclusion.\n"
            "Write for a first-year student: clear language, no unexplained jargon."
        ),
        user=(
            f"Question: {state['question']}\n\n"
            f"Research notes:\n{research_block}"
            f"{critique_context}"
        ),
        max_tokens=800,
    )
    return {
        "draft"    : draft,
        "agent_log": _log("Writer", f"Draft written (revision #{state['revision_count']})"),
    }


# ── Agent 5: Critic ────────────────────────────────────────────────────────────

def critic(state: AssignmentState) -> dict:
    result = _call(
        MODEL_FAST,
        system=(
            "You are a rigorous academic critic. Evaluate the answer and reply in this EXACT format:\n"
            "Score: X/10\n"
            "Strengths: <one sentence>\n"
            "Weaknesses: <one sentence>\n"
            "Verdict: PASS or REVISE"
        ),
        user=f"Question: {state['question']}\n\nAnswer:\n{state['draft']}",
        max_tokens=150,
    )
    try:
        score_match = re.search(r"Score:\s*(\d+)", result)
        score = int(score_match.group(1)) if score_match else 7
    except Exception:
        score = 7

    verdict  = "PASS" if score >= 7 else "REVISE"
    log_msg  = f"Score {score}/10 → {verdict}"

    base = {
        "critique"     : result,
        "quality_score": score,
        "agent_log"    : _log("Critic", log_msg),
    }
    if verdict == "PASS":
        base["final_answer"] = state["draft"]
    return base


# ── Routing ────────────────────────────────────────────────────────────────────

def should_revise(state: AssignmentState) -> Literal["writer", "__end__"]:
    if state["quality_score"] < 7 and state["revision_count"] < 2:
        return "writer"
    return END


print("All 5 agents defined.")

In [ ]:
# ── Build the full graph ───────────────────────────────────────────────────────

def build_assignment_graph():
    g = StateGraph(AssignmentState)

    g.add_node("orchestrator", orchestrator)
    g.add_node("planner",      planner)
    g.add_node("researcher",   researcher)
    g.add_node("writer",       writer)
    g.add_node("critic",       critic)

    g.add_edge(START,          "orchestrator")
    g.add_edge("orchestrator", "planner")
    g.add_edge("planner",      "researcher")
    g.add_edge("researcher",   "writer")
    g.add_edge("writer",       "critic")

    g.add_conditional_edges("critic", should_revise, {
        "writer": "writer",
        END     : END,
    })

    return g.compile()


assignment_app = build_assignment_graph()
print("Full 5-agent graph compiled. Ready.")

In [ ]:
def run_assignment(question: str) -> AssignmentState:
    print("\n" + "═" * 62)
    print("  5-AGENT ASSIGNMENT PLATFORM")
    print(f"  Question: {question}")
    print("═" * 62 + "\n")

    initial: AssignmentState = {
        "question"      : question,
        "route"         : "",
        "subtasks"      : [],
        "research"      : {},
        "draft"         : "",
        "critique"      : "",
        "final_answer"  : "",
        "quality_score" : 0,
        "revision_count": 0,
        "agent_log"     : [],
        "started_at"    : datetime.datetime.now().isoformat(),
    }

    result = assignment_app.invoke(initial)

    print("\n" + "─" * 62)
    print("  PIPELINE SUMMARY")
    print("─" * 62)
    print(f"  Route:     {result['route']}")
    print(f"  Subtasks:  {result['subtasks']}")
    print(f"  Score:     {result['quality_score']}/10")
    print(f"  Revisions: {result['revision_count']}")
    print("\n  CRITIQUE:")
    print(f"  {result['critique']}")
    print("\n  FINAL ANSWER:")
    print("─" * 62)
    print(result["final_answer"])

    return result


# Run it!
result = run_assignment("What are the causes and consequences of deforestation?")

In [ ]:
# TODO: Replace with your own question
YOUR_QUESTION = "Explain the difference between artificial intelligence and machine learning."

result2 = run_assignment(YOUR_QUESTION)

---
# Part 7 — Agent-to-Agent (A2A) Communication

## What is A2A?

So far our agents talk to each other **indirectly** — they share state through LangGraph.
This is one style of A2A communication. Let's map out all the patterns:

```
Pattern 1: Shared State (what we've been doing)
  Agent A writes to state → Agent B reads from state
  ✓ Simple  ✓ Ordered  ✗ Hard to fan out to many agents

Pattern 2: Direct Function Call
  Agent A directly invokes Agent B as a function
  ✓ Explicit  ✓ Synchronous  ✗ Creates tight coupling

Pattern 3: Tool Delegation
  Agent A exposes Agent B as one of its TOOLS
  Agent A calls the tool → Agent B runs → result returned
  ✓ Flexible  ✓ B can be swapped  ✓ Natural fit for LLM tool use

Pattern 4: Message Queue (production systems)
  Agents communicate via a message broker (Redis, Kafka)
  ✓ Decoupled  ✓ Async  ✓ Scalable  ✗ More infrastructure
```

## Pattern 3 in action: Agent as a Tool

We'll build a **Coordinator agent** that can delegate to two specialist agents:
- **FactChecker agent** — verifies factual claims
- **Simplifier agent** — rewrites complex text for beginners

The Coordinator calls them as tools, just like it would call `web_search` or `calculator`.

In [19]:
# ── Specialist agents (these are just functions) ───────────────────────────────

def fact_checker_agent(claim: str) -> str:
    """
    Specialist Agent: verifies a factual claim.
    In a real system this would call web_search.
    """
    response = client.messages.create(
        model=MODEL_FAST,
        max_tokens=200,
        system=(
            "You are a fact-checking agent. Evaluate whether the claim is:"
            " TRUE, FALSE, or UNCERTAIN.\n"
            "Reply in this format:\n"
            "Verdict: TRUE|FALSE|UNCERTAIN\n"
            "Reason: <one sentence>"
        ),
        messages=[{"role": "user", "content": f"Claim: {claim}"}],
    )
    return response.content[0].text


def simplifier_agent(text: str, target_grade: str = "first-year university") -> str:
    """
    Specialist Agent: rewrites complex text for a target audience.
    """
    response = client.messages.create(
        model=MODEL_FAST,
        max_tokens=300,
        system=(
            f"You are a plain-language editor writing for {target_grade} students. "
            "Rewrite the given text: shorter sentences, no jargon, concrete analogies. "
            "Return ONLY the simplified version."
        ),
        messages=[{"role": "user", "content": text}],
    )
    return response.content[0].text


# Quick sanity check
print("=" * 60)
print("  Specialist agent test")
print("=" * 60)
print("\n  FactChecker:")
print(fact_checker_agent("The human brain has exactly 100 billion neurons."))
print("\n  Simplifier:")
print(simplifier_agent(
    "Mitochondria are membrane-bound organelles found in the cytoplasm "
    "of eukaryotic cells that generate most of the cell's ATP via oxidative phosphorylation."
))

  Specialist agent test

  FactChecker:
Verdict: FALSE

Reason: Modern neuroscience research, particularly studies using advanced counting methods, estimates the human brain contains approximately 86 billion neurons, not 100 billion—a figure that has been revised downward from the traditional "100 billion" estimate that was once commonly cited.

  Simplifier:
Mitochondria are tiny structures inside your cells. Think of them like power plants. They take nutrients and convert them into energy (ATP) that your cells can use. This process is called oxidative phosphorylation, but you can just think of it as "energy creation."


In [20]:
# ── Coordinator agent — delegates to specialists via tool use ──────────────────

# Define the specialist agents as tools (this is the A2A bridge)
A2A_TOOLS = [
    {
        "name": "fact_checker",
        "description": (
            "Delegate to the FactChecker specialist agent. "
            "Use this to verify any factual claim before including it in the answer."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"claim": {"type": "string", "description": "The factual claim to verify"}},
            "required": ["claim"],
        },
    },
    {
        "name": "simplifier",
        "description": (
            "Delegate to the Simplifier specialist agent. "
            "Use this to rewrite complex passages into student-friendly language."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "text"        : {"type": "string", "description": "Text to simplify"},
                "target_grade": {"type": "string", "description": "e.g. 'first-year university'"},
            },
            "required": ["text"],
        },
    },
]

A2A_AGENT_REGISTRY = {
    "fact_checker": lambda inp: fact_checker_agent(inp["claim"]),
    "simplifier"  : lambda inp: simplifier_agent(inp["text"], inp.get("target_grade", "first-year university")),
}


def coordinator_agent(user_request: str) -> str:
    """
    Coordinator that delegates to specialist agents via tool use.
    This is A2A communication: Agent → Agent as Tool.
    """
    print("\n" + "═" * 62)
    print("  A2A COORDINATOR AGENT")
    print(f"  Request: {user_request}")
    print("═" * 62)

    messages  = [{"role": "user", "content": user_request}]
    iteration = 0

    while iteration < 8:
        iteration += 1
        response = client.messages.create(
            model=MODEL_GOOD,
            max_tokens=600,
            system=(
                "You are a Coordinator agent. You have two specialist agents available as tools.\n"
                "- fact_checker: verify factual claims before including them\n"
                "- simplifier: rewrite complex text for students\n\n"
                "STRATEGY: For each key claim in your answer, call fact_checker first. "
                "Then simplify any complex passages. Provide a final student-friendly answer."
            ),
            tools=A2A_TOOLS,
            messages=messages,
        )
        messages.append({"role": "assistant", "content": response.content})

        for block in response.content:
            if hasattr(block, "text") and block.text:
                print(f"\n  [Coordinator thinks] {block.text[:150]}")

        if response.stop_reason == "end_turn":
            return next((b.text for b in response.content if hasattr(b, "text")), "")

        # Handle specialist agent calls
        tool_results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            fn  = A2A_AGENT_REGISTRY.get(block.name)
            obs = fn(block.input) if fn else f"Unknown agent: {block.name}"
            print(f"\n  [A2A] Coordinator → {block.name}: {str(block.input)[:80]}")
            print(f"  [A2A] {block.name} → Coordinator: {obs[:120]}")
            tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": obs})

        messages.append({"role": "user", "content": tool_results})

    return "Max iterations reached."


# Run the coordinator
final = coordinator_agent(
    "Explain the greenhouse effect. "
    "Verify any key facts and make sure the final explanation is suitable for first-year students."
)
print("\n" + "═" * 62)
print("  FINAL COORDINATED ANSWER")
print("═" * 62)
print(final)


══════════════════════════════════════════════════════════════
  A2A COORDINATOR AGENT
  Request: Explain the greenhouse effect. Verify any key facts and make sure the final explanation is suitable for first-year students.
══════════════════════════════════════════════════════════════

  [Coordinator thinks] Sure! Let me start by identifying the key claims about the greenhouse effect and verifying them — while also preparing a simplification. I'll fact-che

  [A2A] Coordinator → fact_checker: {'claim': "The greenhouse effect is a natural process where certain gases in Ear
  [A2A] fact_checker → Coordinator: Verdict: TRUE

Reason: The greenhouse effect is indeed a natural atmospheric process where greenhouse gases (carbon diox

  [A2A] Coordinator → fact_checker: {'claim': 'The main greenhouse gases are water vapor, carbon dioxide (CO2), meth
  [A2A] fact_checker → Coordinator: Verdict: TRUE

Reason: Water vapor, carbon dioxide, methane, and nitrous oxide are indeed the four most signi

## What just happened — A2A in plain English

```
User
  │ request
  ▼
Coordinator Agent         ← decides WHAT to do
  ├──[tool: fact_checker]──► FactChecker Agent    ← verifies a claim
  │       observation ◄──────────────────────────
  ├──[tool: simplifier]───► Simplifier Agent      ← rewrites for students
  │       observation ◄──────────────────────────
  └── synthesises final answer
  │
  ▼
User
```

This is **A2A via tool delegation**: the Coordinator treats other agents the same way it treats web_search or a calculator. The specialist agents don't know who called them — they just receive input and return output.

### Why this scales
- Add a new specialist agent? Just add a tool definition — Coordinator needs no changes.
- Replace a specialist? Swap the function in `A2A_AGENT_REGISTRY` — Coordinator still unchanged.
- Run specialists in parallel? LangGraph's parallel node execution handles this (Day 4).

### Exercise 7.1
Add a third specialist: `citation_finder` that returns a made-up APA citation for a topic. Register it in `A2A_AGENT_REGISTRY` and `A2A_TOOLS`. Run the coordinator and ask it to write an essay that cites its sources.

---
## A2A via RPC — Remote Procedure Calls Between Agents

### What is RPC?

**Remote Procedure Call (RPC)** means one process calls a function that runs in *a different process* — potentially on a different machine — and gets the result back as if it were a local call.

In an agentic system, this lets specialist agents run as **independent services** (separate processes, separate servers), each with their own environment, dependencies, and scaling:

```
Without RPC (everything in one process):
  Coordinator → calls fact_checker_agent() → same Python process
  ✓ Simple   ✗ Can't scale independently   ✗ One crash kills all agents

With RPC (agents as services):
  Coordinator ──HTTP/gRPC──► FactChecker Service  (own process / container)
                              Simplifier Service   (own process / container)
  ✓ Independent scaling   ✓ Fault isolation   ✓ Any language on either side
```

### The three common RPC styles in agentic systems

| Style | Protocol | When to use |
|-------|----------|-------------|
| **XML-RPC** | HTTP + XML | Quick prototypes, built into Python stdlib |
| **JSON-RPC** | HTTP + JSON | Modern REST-style, easy to debug with curl |
| **gRPC** | HTTP/2 + Protobuf | Production microservices, high throughput |

We'll demonstrate **XML-RPC** because it requires zero extra installs — it's in Python's standard library — and the concept transfers directly to gRPC or any other style.

In [21]:
import threading
from xmlrpc.server import SimpleXMLRPCServer
from xmlrpc.client import ServerProxy

# ── Step 1: Define the specialist agent as an RPC-exposed service ──────────────
# These are the same agent functions from above — nothing changes.
# We're just making them accessible over the network.

RPC_PORT = 8901

def rpc_fact_check(claim: str) -> str:
    """RPC-exposed: verify a factual claim."""
    return fact_checker_agent(claim)

def rpc_simplify(text: str) -> str:
    """RPC-exposed: rewrite text for students."""
    return simplifier_agent(text)


# ── Step 2: Start the specialist agent server in a background thread ───────────
# In production this would be a separate process (or container).
# Here we use a daemon thread so it stops when the notebook kernel stops.

def _start_agent_rpc_server():
    server = SimpleXMLRPCServer(
        ("127.0.0.1", RPC_PORT),
        logRequests=False,   # silence per-request prints
        allow_none=True,
    )
    server.register_function(rpc_fact_check, "fact_check")
    server.register_function(rpc_simplify,   "simplify")
    print(f"  [AgentServer] Listening on port {RPC_PORT}...")
    server.serve_forever()

server_thread = threading.Thread(target=_start_agent_rpc_server, daemon=True)
server_thread.start()
time.sleep(0.4)   # give the server a moment to start

print("Agent RPC server started.")

  [AgentServer] Listening on port 8901...
Agent RPC server started.


In [22]:
# ── Step 3: Coordinator calls the remote agents via RPC ───────────────────────
# From the coordinator's perspective this looks like a local function call.
# Under the hood it's an HTTP request to the server running in the other thread.

remote_agents = ServerProxy(f"http://127.0.0.1:{RPC_PORT}/")

print("=" * 62)
print("  A2A VIA RPC — Coordinator calling remote specialist agents")
print("=" * 62)

# Call 1: fact-check over RPC
claim = "The speed of light in a vacuum is approximately 300,000 km/s."
print(f"\n  Coordinator → [RPC] → FactChecker")
print(f"  Claim: {claim}")
fc_result = remote_agents.fact_check(claim)
print(f"  FactChecker → [RPC] → Coordinator: {fc_result}")

# Call 2: simplify over RPC
complex_text = (
    "Photosynthesis is the biochemical process by which chlorophyll-containing "
    "organisms transduce electromagnetic radiation from the sun into chemical "
    "potential energy stored in glucose via the Calvin cycle."
)
print(f"\n  Coordinator → [RPC] → Simplifier")
print(f"  Text: {complex_text[:80]}...")
simplified = remote_agents.simplify(complex_text)
print(f"  Simplifier → [RPC] → Coordinator:\n  {simplified}")

print()
print("=" * 62)
print("  KEY POINT")
print("=" * 62)
print("""
  remote_agents.fact_check(claim)
      ↑ looks like a plain Python function call

  Under the hood:
    Coordinator serialises the args to XML
    → HTTP POST to 127.0.0.1:8901
    → Server deserialises, calls rpc_fact_check()
    → rpc_fact_check() calls fact_checker_agent() (Claude)
    → Result serialised back as XML
    → Coordinator deserialises and returns the string

  In production:
    Replace 127.0.0.1:8901 with agent-service.internal:8901
    Replace XML-RPC with gRPC + Protobuf for performance
    Add auth, retry logic, and circuit breakers
    Each agent becomes its own Docker container / K8s pod
""")

  A2A VIA RPC — Coordinator calling remote specialist agents

  Coordinator → [RPC] → FactChecker
  Claim: The speed of light in a vacuum is approximately 300,000 km/s.
  FactChecker → [RPC] → Coordinator: Verdict: TRUE

Reason: The speed of light in a vacuum is approximately 299,792 km/s, which rounds to 300,000 km/s, making this a commonly accepted approximation used in scientific contexts.

  Coordinator → [RPC] → Simplifier
  Text: Photosynthesis is the biochemical process by which chlorophyll-containing organi...
  Simplifier → [RPC] → Coordinator:
  Plants use sunlight to make sugar. They capture the sun's energy with a green pigment called chlorophyll. This energy powers a series of chemical reactions that build sugar molecules. The plant then uses this sugar as food.

  KEY POINT

  remote_agents.fact_check(claim)
      ↑ looks like a plain Python function call

  Under the hood:
    Coordinator serialises the args to XML
    → HTTP POST to 127.0.0.1:8901
    → Server deseriali

### RPC vs. Tool Delegation — which to use?

```
Tool Delegation (Pattern 3, shown earlier):
  All agents live in the same Python process.
  The Coordinator calls specialist functions directly.
  ✓ Simple to code   ✓ Great for prototypes and this bootcamp
  ✗ One process crash = all agents down

RPC (this section):
  Each specialist agent runs as an independent process / server.
  The Coordinator calls it over HTTP/gRPC as if it were local.
  ✓ Independent deployment   ✓ Independent scaling
  ✓ Agents can be in different languages (Python, Go, Node...)
  ✓ Agents can run on different machines
  ✗ More infrastructure to manage

Rule of thumb:
  Prototype  → Tool delegation (everything in one process)
  Production → RPC / microservices (each agent as a service)
```

### A2A in the real world — Google's A2A Protocol

In 2025 Google open-sourced the **Agent-to-Agent (A2A) Protocol** — a standardised HTTP+JSON spec for how any two agents, built by different teams or companies, can discover each other and communicate. It's the RPC pattern generalised into an open standard, similar to what MCP did for tools.

The flow looks exactly like what you just built:
1. Each agent publishes an **Agent Card** (JSON describing capabilities)
2. Other agents discover the card and call its endpoints via HTTP
3. Results flow back as structured JSON

Day 4 introduces MCP (the tool side of this). By Day 5 you'll understand why both protocols exist and when to use each.

![gRPC Diagram](https://enouvo.com/wp-content/uploads/2020/08/Compare-gprc-rest-fix-01-1-1024x847.jpg)

---
## Challenge Exercises (if you finish early)

**Level 1 — Add a memory node**  
Add a `memory_node` that runs AFTER the Critic. It appends the question and final answer to a Python list. Use `session_memory = []` at the module level and have the node append to it. Print the memory contents after 2 runs.

**Level 2 — Parallel researcher**  
Instead of the Researcher calling `react_loop` for each subtask sequentially, run them concurrently using `concurrent.futures.ThreadPoolExecutor`:

```python
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {subtask: ex.submit(react_loop, subtask) for subtask in state["subtasks"]}
    findings = {task: f.result() for task, f in futures.items()}
```

Measure the speedup.

**Level 3 — Self-improving Orchestrator**  
After the Critic returns a score, feed the critique back into the Orchestrator. Let it re-route to a different path if the original route produced a low score. (Hint: add a `retry_route` conditional edge.)

---
# Part 8 — Preview: What's Coming in Day 4

## Where we are today

```
Day 3 system:

  ┌─────────────────────────────────────────────────┐
  │           5-Agent LangGraph Pipeline             │
  │  Orchestrator → Planner → Researcher → Writer   │
  │                                  ↑              │
  │                            Critic (loop)        │
  │                                                 │
  │  Memory: ❌ (lost when notebook restarts)        │
  │  Tools:  ✅ web_search, calculator               │
  │  A2A:    ✅ via shared state + tool delegation   │
  └─────────────────────────────────────────────────┘
```

## Day 4 upgrades every layer

```
Day 4 system:

  ┌─────────────────────────────────────────────────┐
  │         Same 5-Agent LangGraph Pipeline          │
  │                                                 │
  │  Memory: ✅ ChromaDB vector store               │
  │    → Agents remember past Q&As across restarts  │
  │    → Researcher can query previous findings     │
  │                                                 │
  │  RAG:    ✅ Retrieval-Augmented Generation       │
  │    → Researcher searches a knowledge base       │
  │    → Grounds answers in real documents          │
  │                                                 │
  │  Tools:  ✅ MCP (Model Context Protocol)        │
  │    → Tools are now server processes             │
  │    → Any agent can use any MCP tool             │
  │                                                 │
  │  Multi-agent coordination: ✅ Parallel nodes    │
  │    → Researcher subtasks run simultaneously     │
  │    → 3× faster on multi-subtask questions       │
  └─────────────────────────────────────────────────┘
```

## What is RAG?

**Retrieval-Augmented Generation** gives agents access to a searchable knowledge base:

```
Without RAG:  Question → Agent (uses training knowledge only) → Answer

With RAG:     Question → Retriever (searches vector DB) → Top-K relevant chunks
                                                              ↓
              Question + retrieved context → Agent → Grounded answer
```

The key insight: **agents can now answer questions about YOUR documents** — internal reports, lecture notes, company data — not just what the LLM was trained on.

```python
# Day 4 preview — ChromaDB memory
import chromadb

chroma = chromadb.PersistentClient(path="./memory")   # survives restarts
collection = chroma.get_or_create_collection("assignments")

# Save a Q&A to memory
collection.add(
    documents=[answer],
    metadatas=[{"question": question, "score": quality_score}],
    ids=[f"qa_{timestamp}"],
)

# Later — retrieve similar past questions
results = collection.query(query_texts=[new_question], n_results=3)
# results["documents"] → the 3 most relevant past answers
```

## What is MCP?

**Model Context Protocol** (Anthropic, 2024) standardises how agents connect to tools:

```
Before MCP:   Each tool is a custom Python function in your code
              Changing a tool = editing the agent's source code

With MCP:     Tools are standalone server processes
              Agent connects via a standard protocol (HTTP/stdio)
              Swap or add tools without touching agent code
```

Day 4 turns the `web_search` and `calculator` functions into MCP servers, so any agent in the graph can use them transparently.

## Day 5 preview

Day 5 wraps everything in a production interface:

```
Student types a question
        ↓
  WhatsApp / Web UI
        ↓
  FastAPI backend
        ↓
  5-agent LangGraph pipeline (Day 3)
  + ChromaDB memory (Day 4)
  + MCP tools (Day 4)
        ↓
  Answer streamed back
        ↓
  Saved to memory for next session
```

> **The whole bootcamp is one system, built incrementally.**  
> Day 3's graph becomes Day 4's graph with memory.  
> Day 4's graph becomes Day 5's backend.  
> You're not learning isolated topics — you're building one thing.

In [ ]:
# ── A taste of Day 4: in-memory vector search (no ChromaDB install needed) ─────
#
# Real Day 4 uses ChromaDB (persistent) and sentence-transformers (semantic).
# This toy version uses simple keyword matching to show the concept.

import math as _math

class ToyVectorStore:
    """A tiny in-memory store to preview RAG mechanics."""

    def __init__(self):
        self._docs: list[dict] = []

    def add(self, text: str, metadata: dict = None) -> None:
        self._docs.append({"text": text, "meta": metadata or {}})
        print(f"  [VectorStore] Saved: {text[:60]}...")

    def search(self, query: str, n: int = 2) -> list[str]:
        query_words = set(query.lower().split())
        scored = []
        for doc in self._docs:
            doc_words = set(doc["text"].lower().split())
            overlap   = len(query_words & doc_words)
            if overlap:
                scored.append((overlap, doc["text"]))
        scored.sort(reverse=True)
        return [text for _, text in scored[:n]]


memory_store = ToyVectorStore()

# Simulate the agent saving past answers to memory
past_qa = [
    ("What is photosynthesis?",
     "Photosynthesis is the process by which plants convert sunlight, water, "
     "and CO2 into glucose and oxygen. It happens in the chloroplasts."),
    ("How does the water cycle work?",
     "The water cycle involves evaporation, condensation, and precipitation. "
     "Solar energy drives evaporation from oceans and lakes."),
    ("What causes climate change?",
     "Climate change is driven by greenhouse gas emissions (CO2, methane) "
     "that trap heat in the atmosphere, raising global temperatures."),
]

print("Saving past Q&As to memory...\n")
for q, a in past_qa:
    memory_store.add(f"Q: {q}\nA: {a}", metadata={"question": q})

# Now simulate using memory in a new query
new_question = "How does CO2 affect plant growth and global temperature?"
retrieved    = memory_store.search(new_question, n=2)

print(f"\nNew question: {new_question}")
print(f"\nRetrieved from memory:")
for i, doc in enumerate(retrieved, 1):
    print(f"  [{i}] {doc[:100]}...")

# Use retrieved context in the answer
context_block = "\n\n".join(retrieved)
response = client.messages.create(
    model=MODEL_FAST, max_tokens=300,
    system=(
        "You are an academic assistant. Use the provided context from memory "
        "to answer the question. If context is relevant, cite it naturally."
    ),
    messages=[{"role": "user", "content":
        f"Context from memory:\n{context_block}\n\nQuestion: {new_question}"}],
)

print("\n" + "═" * 60)
print("  RAG-GROUNDED ANSWER (uses retrieved memory)")
print("═" * 60)
print(response.content[0].text)

print("""
  ← On Day 4 this becomes:
    - ChromaDB with persistent storage (survives restarts)
    - Real semantic embeddings (meaning-based search, not keyword)
    - The Researcher node queries memory BEFORE calling web_search
    - If memory has a recent answer → skip web search → save cost
""")

---
# Checkpoint — Demo to the Class

Run the cell below. Show:
1. The 5-agent pipeline executing (all agent logs printing)
2. The route the Orchestrator chose
3. The subtasks the Planner created
4. The Critic's score
5. The final answer
6. (Bonus) The A2A coordinator calling a specialist

In [ ]:
# Checkpoint demo
checkpoint_result = run_assignment(
    "What is blockchain technology and why is it considered secure?"
)

print("\n" + "─" * 62)
print("  FULL AGENT LOG")
print("─" * 62)
for entry in checkpoint_result["agent_log"]:
    print(f"  {entry}")

---
# Day 3 Summary

## What you built
- Revisited the **ReAct loop** and embedded it inside a LangGraph node
- Wrapped the **Wikipedia REST API** as a hosted tool — no building, just adapting
- Understood **why one agent isn't enough** and what specialisation buys you
- Built a **2-node LangGraph graph** with shared state
- Added **conditional routing** — Orchestrator classifies, Critic loops
- Implemented **clean agent handoffs** using `Annotated` accumulators
- Assembled the full **5-agent pipeline**: Orchestrator → Planner → Researcher → Writer → Critic
- Demonstrated **A2A via tool delegation** (Agent as Tool pattern)
- Demonstrated **A2A via XML-RPC** — agents calling each other as remote services
- Previewed **Day 4**: RAG, ChromaDB memory, MCP, and parallel nodes

## Key vocabulary

| Term | Definition |
|------|------------|
| LangGraph | Framework for building agent pipelines as directed state machines |
| State dict | Shared notepad that flows through all nodes in the graph |
| Node | A Python function in the graph — one agent's responsibility |
| Edge | A connection between nodes — static or conditional |
| Conditional edge | A routing function that decides the next node based on state |
| Handoff | When one agent finishes and the next reads its output from state |
| Hosted tool | A third-party service (Wikipedia, DuckDuckGo) wrapped as an agent tool |
| Thin adapter | The ~10-line wrapper that connects a hosted API to your agent |
| A2A | Agent-to-Agent communication — agents calling each other |
| RPC | Remote Procedure Call — calling a function in a different process over the network |
| XML-RPC | Simplest RPC style, built into Python stdlib |
| gRPC | Production RPC — faster, typed, used in real microservice architectures |
| RAG | Retrieval-Augmented Generation — grounding answers in a knowledge base |
| MCP | Model Context Protocol — standardised tool/server connectivity |
| Annotated | LangGraph type hint for fields that should be accumulated, not overwritten |

## What's next — Day 4

> The Day 3 pipeline is stateless — restart the notebook and it forgets everything.  
> Day 4 upgrades memory from a Python list to **ChromaDB**, a vector database that:
> - Persists across sessions
> - Finds *semantically similar* past answers (not just exact matches)
> - Powers RAG — agents answer questions about your own documents
>
> We also wrap tools in **MCP servers** so any agent in the graph can call any tool transparently.
> The RPC pattern you built here directly informs how MCP servers communicate.

---
*Agentic AI Bootcamp — Day 3 complete*